# Qwen3.5-4B function-handoff ultra-simple v4 TEC experiment

This notebook runs the ultra-simple function-handoff full LLM multi-agent revision. Inter-role handoff and TEC tool use still share one `<tool_call>` function-call block.

Revision v4 removes free-text assignments from worker behavior, renders every LLM turn stateless, and puts compact runtime facts plus state-gated recipes in front of Qwen3.5-4B.

This notebook now runs the full five-task benchmark by default. Use `RUN_ALL_TASKS=False` only for a single-task debug pilot.


## 1. Clean Colab setup


In [ ]:
# ============================================================
# Repair Qwen3.5 Transformers environment in Colab
# ============================================================

import subprocess
import sys

def run(cmd):
    print("$", " ".join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], check=True)

print("Removing incompatible Transformers installation...")
run([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "transformers",
])

print("\nInstalling current Transformers main with Qwen3.5 support...")
run([
    sys.executable, "-m", "pip", "install",
    "-q",
    "--no-cache-dir",
    "--force-reinstall",
    "transformers @ git+https://github.com/huggingface/transformers.git@main",
])

print("\nInstalling Qwen3.5 vision/import dependencies...")
run([
    sys.executable, "-m", "pip", "install",
    "-q",
    "--no-cache-dir",
    "--force-reinstall",
    "Pillow==12.2.0",
    "torchvision",
    "accelerate",
    "safetensors",
    "huggingface_hub",
])

print(
    "\nEnvironment repaired.\n"
    "Now restart the Colab runtime once:\n"
    "Runtime -> Restart session\n"
    "Then rerun cells from imports/configuration onward."
)

After the setup cell finishes, restart the Colab runtime once before loading the model.


## 2. Import check


In [ ]:
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


## 3. Clone or update repository


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/tec_agent_project")
REPO_URL = "https://github.com/ilyakosilov/tec_agent_project.git"

if PROJECT_DIR.exists() and (PROJECT_DIR / ".git").exists():
    tracked_changed = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "diff", "--quiet"],
        check=False,
    ).returncode != 0
    staged_changed = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "diff", "--cached", "--quiet"],
        check=False,
    ).returncode != 0
    if tracked_changed or staged_changed:
        raise RuntimeError("Tracked local changes exist in the Colab checkout.")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif PROJECT_DIR.exists():
    raise RuntimeError(f"{PROJECT_DIR} exists but is not a git checkout.")
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print("Working directory:", Path.cwd())


## 4. Project imports


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

from tec_agents.agents.llm_multi_agent_function_handoff import (
    LLMFunctionHandoffMultiAgent,
)
from tec_agents.agents.llm_multi_agent_function_handoff_prompts import (
    ARCHITECTURE_NAME,
    PROMPT_REVISION,
)
from tec_agents.agents.llm_multi_agent_function_handoff_protocol import (
    FUNCTION_HANDOFF_PROTOCOL_VERSION,
)
from tec_agents.data.datasets import get_dataset_summary, register_dataset
from tec_agents.eval.five_task_configs import (
    get_five_task_configs,
    build_five_task_eval_task,
    build_five_task_expected_sequence,
)
from tec_agents.eval.gold_runner import GoldRunner
from tec_agents.eval.metrics import MetricResult, compare_agent_to_gold
from tec_agents.eval.task_set import task_to_dict
from tec_agents.llm.local_qwen import LocalQwenChatModel
from tec_agents.mcp.client import LocalMCPClient
from tec_agents.mcp.server import build_local_mcp_server


## 5. CONFIG


In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B"
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
TORCH_DTYPE = "float16"
MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
DO_SAMPLE = False

DATASET_REF = "default"
DATASET_FILENAME = "tec_regions_2024_01_01_to_2024_04_01_hourly.parquet"
PROCESSED_DATASET_PATH = PROJECT_DIR / "data" / "processed" / DATASET_FILENAME
START = "2024-03-01"
END = "2024-04-01"

ARCHITECTURE_MODE = "qwen_multi_agent_function_handoff_full_llm"
FUNCTION_HANDOFF_PROTOCOL_VERSION = "function_handoff_v2"
PROMPT_REVISION = "function_handoff_ultra_simple_state_recipes_v4"
EXPERIMENT_ID = "qwen_multi_agent_function_handoff_ultra_simple_v4_batch_colab"

MAX_ORCHESTRATION_STEPS = 12
MAX_ROLE_STEPS = 8
MAX_TOOL_CALLS_PER_ROLE = 8
MAX_PARSE_RETRIES = 2
MAX_TOOL_RETRIES = 2
STATE_FEEDBACK_MODE = "function_handoff_state"

RUN_ALL_TASKS = True
SELECTED_PRESET = "hightec_midlat_europe"

OUTPUT_ROOT = PROJECT_DIR / "outputs" / "metrics" / "real_runs" / "multi_agent"
EXPERIMENT_OUTPUT_DIR = OUTPUT_ROOT / "experiment_8_function_handoff_ultra_simple_v4"
OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_OUTPUT_PATH = OUTPUT_DIR / "qwen_multi_agent_function_handoff_ultra_simple_v4_batch_colab.json"
PER_TASK_OUTPUT_TEMPLATE = "qwen_multi_agent_function_handoff_ultra_simple_v4_{preset_id}_colab.json"

TASK_CONFIGS = get_five_task_configs(dataset_ref=DATASET_REF)
if RUN_ALL_TASKS:
    SELECTED_TASK_CONFIGS = TASK_CONFIGS
else:
    SELECTED_TASK_CONFIGS = [c for c in TASK_CONFIGS if c["preset_id"] == SELECTED_PRESET]
assert SELECTED_TASK_CONFIGS, f"Unknown SELECTED_PRESET={SELECTED_PRESET!r}"

print("Architecture:", ARCHITECTURE_MODE)
print("Protocol:", FUNCTION_HANDOFF_PROTOCOL_VERSION)
print("Prompt revision:", PROMPT_REVISION)
print("Experiment id:", EXPERIMENT_ID)
print("Selected tasks:", [c["preset_id"] for c in SELECTED_TASK_CONFIGS])
print("Output dir:", OUTPUT_DIR)


## Planned test questions

This preview is for the human notebook reader only. Expected tool sequences are used later for evaluation and are not passed into the LLM agents.


In [ ]:
preview_rows = []
for config in SELECTED_TASK_CONFIGS:
    preview_rows.append({
        "preset_id": config["preset_id"],
        "task_type": config["task_type"],
        "query": config["query"],
        "expected_tool_sequence": " -> ".join(build_five_task_expected_sequence(config)),
    })
pd.DataFrame(preview_rows)


## 6. Dataset setup


In [ ]:
DATASET_PATH = PROCESSED_DATASET_PATH
assert DATASET_PATH.exists(), f"Missing dataset: {DATASET_PATH}"
register_dataset(dataset_ref=DATASET_REF, path=DATASET_PATH, file_format="parquet")
get_dataset_summary(DATASET_REF)


## 7. Qwen3.5-4B loading


In [ ]:
model = LocalQwenChatModel(
    model_name=MODEL_NAME,
    device_map="auto",
    torch_dtype=TORCH_DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_8bit=LOAD_IN_8BIT,
    trust_remote_code=True,
    max_input_tokens=MAX_INPUT_TOKENS,
)
if hasattr(model, "tokenizer"):
    model.tokenizer.truncation_side = "left"
    print("Tokenizer truncation_side:", model.tokenizer.truncation_side)
print("Loaded model:", MODEL_NAME)


## 8. Helpers


In [ ]:
def to_jsonable(value):
    if hasattr(value, "to_dict"):
        return to_jsonable(value.to_dict())
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if hasattr(value, "item"):
        return value.item()
    return value


def metric_dict(metric_result):
    if metric_result is None:
        return {}
    return metric_result.metrics if isinstance(metric_result, MetricResult) else dict(metric_result)


def final_answer_preview(result, limit=360):
    text = getattr(result, "answer", "") or ""
    return text[:limit] + ("..." if len(text) > limit else "")


def build_verdict_checks(result, metric_result):
    metrics = metric_dict(metric_result)
    return {
        "agent_success": bool(result.success),
        "workflow_completed": bool(getattr(result, "workflow_completed", False)),
        "terminal_numeric_artifact_present": bool(getattr(result, "terminal_numeric_artifact_present", False)),
        "tool_sequence_match": metrics.get("tool_sequence_match"),
        "final_answer_present": bool(result.answer),
        "no_parse_errors": result.parse_error_count == 0,
        "no_stall": not result.stalled_loop_detected,
        "no_forbidden_function_calls": result.forbidden_function_call_count == 0,
    }


def overall_ok_from_checks(checks):
    required = [v for v in checks.values() if isinstance(v, bool)]
    return bool(required) and all(required)


def key_metric_summary(task_type, metrics):
    if task_type == "high_tec":
        return f"threshold_abs_error={metrics.get('threshold_abs_error')}; interval_count_error={metrics.get('interval_count_error')}"
    if task_type == "stable_intervals":
        return f"stable_interval_count_error={metrics.get('stable_interval_count_error')}"
    if task_type == "compare_regions":
        return f"mean_abs_error_avg={metrics.get('mean_abs_error_avg')}; pairwise_delta_count_match={metrics.get('pairwise_delta_count_match')}"
    if task_type == "report":
        return f"required_artifacts_present={metrics.get('required_artifacts_present')}; grounded={metrics.get('report_grounded_in_artifacts')}"
    return ""


def diagnostic_rate(records, key):
    values = []
    for record in records:
        for item in ((record.get("agent_result") or {}).get("llm_call_diagnostics") or []):
            value = item.get(key)
            if isinstance(value, bool):
                values.append(value)
    return None if not values else sum(values) / len(values)


def diagnostic_avg(records, key):
    values = []
    for record in records:
        for item in ((record.get("agent_result") or {}).get("llm_call_diagnostics") or []):
            value = item.get(key)
            if isinstance(value, (int, float)) and not isinstance(value, bool):
                values.append(value)
    return None if not values else sum(values) / len(values)


## 9. Batch loop


In [ ]:
all_results = []
gold_runner = GoldRunner()

for task_config in SELECTED_TASK_CONFIGS:
    preset_id = task_config["preset_id"]
    query = task_config["query"]
    expected_sequence = build_five_task_expected_sequence(task_config)
    eval_task = build_five_task_eval_task(task_config)

    print()
    print("===", preset_id, "===")
    server = build_local_mcp_server(run_id=f"qwen_multi_agent_function_handoff_ultra_simple_v4_{preset_id}_colab")
    client = LocalMCPClient(server)
    agent = LLMFunctionHandoffMultiAgent(
        model=model,
        client=client,
        max_orchestration_steps=MAX_ORCHESTRATION_STEPS,
        max_role_steps=MAX_ROLE_STEPS,
        max_tool_calls_per_role=MAX_TOOL_CALLS_PER_ROLE,
        max_parse_retries=MAX_PARSE_RETRIES,
        max_tool_retries=MAX_TOOL_RETRIES,
        temperature=TEMPERATURE,
        state_feedback_mode=STATE_FEEDBACK_MODE,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    result = agent.run(query)
    result_dict = result.to_dict()

    gold = gold_runner.run(eval_task)
    metric_result = None
    if gold.status == "success" and gold.result is not None:
        agent_metric_payload = dict(result.tool_results)
        agent_metric_payload.update({
            "parse_error_count": result.parse_error_count,
            "invalid_json_count": result.invalid_json_count,
            "invalid_function_name_count": result.invalid_function_name_count,
            "forbidden_function_call_count": result.forbidden_function_call_count,
            "repeated_tool_call_count": result.repeated_tool_call_count,
            "multiple_function_blocks_in_single_output_count": result.multiple_function_blocks_in_single_output_count,
            "multiple_protocol_blocks_in_single_output_count": result.multiple_protocol_blocks_in_single_output_count,
            "invalid_artifact_handle_count": result.invalid_artifact_handle_count,
            "stalled_loop_detected": result.stalled_loop_detected,
        })
        metric_result = compare_agent_to_gold(
            task_id=eval_task.task_id,
            task_type=eval_task.task_type,
            agent_result=agent_metric_payload,
            gold_result=gold.result,
            agent_trace=result.trace,
            task=task_to_dict(eval_task),
            parsed_task=result.parsed_task,
            orchestration_steps=result.orchestration_steps,
        )

    metrics = metric_dict(metric_result)
    numeric_match_with_gold = bool(metric_result.success) if metric_result is not None else None
    result_dict["numeric_match_with_gold"] = numeric_match_with_gold
    verdict_checks = build_verdict_checks(result, metric_result)
    overall_ok = overall_ok_from_checks(verdict_checks)

    record = {
        "selected_preset": preset_id,
        "task_config": task_config,
        "query": query,
        "expected_tool_sequence": expected_sequence,
        "actual_tool_sequence": result.actual_tool_sequence,
        "attempted_tec_tool_sequence": result.attempted_tec_tool_sequence,
        "successful_tec_tool_sequence": result.successful_tec_tool_sequence,
        "skipped_repeated_tec_tool_calls": result.skipped_repeated_tec_tool_calls,
        "failed_tec_tool_calls": result.failed_tec_tool_calls,
        "architecture": ARCHITECTURE_MODE,
        "model_name": MODEL_NAME,
        "function_handoff_protocol_version": FUNCTION_HANDOFF_PROTOCOL_VERSION,
        "prompt_revision": PROMPT_REVISION,
        "agent_result": result_dict,
        "gold_status": gold.status,
        "gold_error": gold.error,
        "gold_result": gold.result,
        "metrics": metrics,
        "verdict_checks": verdict_checks,
        "overall_ok": overall_ok,
        "success": bool(result.success),
        "workflow_completed": bool(result.workflow_completed),
        "terminal_numeric_artifact_present": bool(result.terminal_numeric_artifact_present),
        "numeric_match_with_gold": numeric_match_with_gold,
        "final_answer_preview": final_answer_preview(result),
        "role_agent_order": result.role_agent_order,
        "role_call_sequence": result.role_call_sequence,
        "handoff_function_sequence": result.handoff_function_sequence,
        "orchestrator_decisions": result.orchestrator_decisions,
        "function_calls": result.function_calls,
        "role_outputs": result.role_outputs,
        "tool_observations": result.tool_observations,
        "llm_call_diagnostics": result.llm_call_diagnostics,
        "invalid_function_name_count": result.invalid_function_name_count,
        "forbidden_function_call_count": result.forbidden_function_call_count,
        "parse_error_count": result.parse_error_count,
        "invalid_json_count": result.invalid_json_count,
        "repeated_tool_call_count": result.repeated_tool_call_count,
        "multiple_function_blocks_in_single_output_count": result.multiple_function_blocks_in_single_output_count,
        "multiple_protocol_blocks_in_single_output_count": result.multiple_protocol_blocks_in_single_output_count,
        "invalid_artifact_handle_count": result.invalid_artifact_handle_count,
        "repeated_role_message_count": result.repeated_role_message_count,
        "orchestrator_equivalent_assignment_without_state_change_count": result.orchestrator_equivalent_assignment_without_state_change_count,
        "orchestrator_free_text_ignored_count": result.orchestrator_free_text_ignored_count,
        "successful_final_tool_without_return_count": result.successful_final_tool_without_return_count,
        "data_agent_repeated_retrieval_after_all_assignment_series_present_count": result.data_agent_repeated_retrieval_after_all_assignment_series_present_count,
        "math_repeated_tool_after_terminal_artifact_count": result.math_repeated_tool_after_terminal_artifact_count,
        "math_repeated_tool_after_new_intermediate_artifact_count": result.math_repeated_tool_after_new_intermediate_artifact_count,
        "math_failed_to_return_after_assignment_artifact_present_count": result.math_failed_to_return_after_assignment_artifact_present_count,
        "math_recomputed_existing_region_stats_count": result.math_recomputed_existing_region_stats_count,
        "math_assignment_completed_but_not_returned_count": result.math_assignment_completed_but_not_returned_count,
        "tool_error_count": result.tool_error_count,
        "stalled_loop_detected": result.stalled_loop_detected,
        "repair_attempt_count": result.repair_attempt_count,
        "retry_count": result.retry_count,
        "analysis_agent_called": result.analysis_agent_called,
        "report_agent_called": result.report_agent_called,
        "findings_present": result.findings_present,
        "failure_reason": result.failure_reason,
        "key_metric_summary": key_metric_summary(task_config["task_type"], metrics),
    }

    per_task_path = OUTPUT_DIR / PER_TASK_OUTPUT_TEMPLATE.format(preset_id=preset_id)
    with per_task_path.open("w", encoding="utf-8") as f:
        json.dump(to_jsonable(record), f, ensure_ascii=False, indent=2)
    print("Saved:", per_task_path)
    print("success:", record["success"], "workflow:", record["workflow_completed"], "terminal:", record["terminal_numeric_artifact_present"], "tools:", result.actual_tool_sequence)
    all_results.append(record)


## 10. Summary table and save batch


In [ ]:
def rate(values):
    values = [v for v in values if isinstance(v, bool)]
    return None if not values else sum(values) / len(values)


def avg(records, key):
    values = [r.get(key) for r in records if isinstance(r.get(key), (int, float))]
    return None if not values else sum(values) / len(values)


summary = {
    "n_tasks": len(all_results),
    "n_success": sum(1 for r in all_results if r["success"]),
    "n_failed": sum(1 for r in all_results if not r["success"]),
    "success_rate": rate([r["success"] for r in all_results]),
    "n_overall_ok": sum(1 for r in all_results if r["overall_ok"]),
    "overall_ok_rate": rate([r["overall_ok"] for r in all_results]),
    "workflow_completed_rate": rate([r["workflow_completed"] for r in all_results]),
    "terminal_numeric_artifact_present_rate": rate([r["terminal_numeric_artifact_present"] for r in all_results]),
    "final_answer_present_rate": rate([bool((r["agent_result"] or {}).get("answer")) for r in all_results]),
    "analysis_agent_called_rate": rate([r["analysis_agent_called"] for r in all_results]),
    "report_agent_called_rate": rate([r["report_agent_called"] for r in all_results]),
    "tool_sequence_match_rate": rate([(r.get("metrics") or {}).get("tool_sequence_match") for r in all_results]),
    "role_agent_order_match_rate": rate([(r.get("metrics") or {}).get("role_agent_order_match") for r in all_results]),
    "artifact_flow_valid_rate": rate([(r.get("metrics") or {}).get("artifact_flow_valid") for r in all_results]),
    "required_role_agents_called_rate": rate([(r.get("metrics") or {}).get("required_role_agents_called") for r in all_results]),
    "avg_tool_call_count": sum(len(r["actual_tool_sequence"]) for r in all_results) / max(1, len(all_results)),
    "avg_attempted_tec_tool_call_count": sum(len(r["attempted_tec_tool_sequence"]) for r in all_results) / max(1, len(all_results)),
    "avg_tool_error_count": avg(all_results, "tool_error_count"),
    "avg_parse_error_count": avg(all_results, "parse_error_count"),
    "avg_invalid_function_name_count": avg(all_results, "invalid_function_name_count"),
    "avg_forbidden_function_call_count": avg(all_results, "forbidden_function_call_count"),
    "avg_repeated_tool_call_count": avg(all_results, "repeated_tool_call_count"),
    "avg_invalid_artifact_handle_count": avg(all_results, "invalid_artifact_handle_count"),
    "avg_repeated_role_message_count": avg(all_results, "repeated_role_message_count"),
    "avg_orchestrator_equivalent_assignment_without_state_change_count": avg(all_results, "orchestrator_equivalent_assignment_without_state_change_count"),
    "avg_orchestrator_free_text_ignored_count": avg(all_results, "orchestrator_free_text_ignored_count"),
    "avg_successful_final_tool_without_return_count": avg(all_results, "successful_final_tool_without_return_count"),
    "avg_data_agent_repeated_retrieval_after_all_assignment_series_present_count": avg(all_results, "data_agent_repeated_retrieval_after_all_assignment_series_present_count"),
    "avg_math_repeated_tool_after_terminal_artifact_count": avg(all_results, "math_repeated_tool_after_terminal_artifact_count"),
    "avg_math_repeated_tool_after_new_intermediate_artifact_count": avg(all_results, "math_repeated_tool_after_new_intermediate_artifact_count"),
    "avg_math_failed_to_return_after_assignment_artifact_present_count": avg(all_results, "math_failed_to_return_after_assignment_artifact_present_count"),
    "avg_math_recomputed_existing_region_stats_count": avg(all_results, "math_recomputed_existing_region_stats_count"),
    "avg_math_assignment_completed_but_not_returned_count": avg(all_results, "math_assignment_completed_but_not_returned_count"),
    "prompt_truncation_rate": diagnostic_rate(all_results, "prompt_was_truncated"),
    "avg_state_character_count": diagnostic_avg(all_results, "state_character_count"),
    "avg_state_history_items_included": diagnostic_avg(all_results, "state_history_items_included"),
    "raw_output_prefix_before_tool_call_rate": diagnostic_rate(all_results, "raw_output_had_prefix_before_tool_call"),
    "output_cleaning_applied_rate": diagnostic_rate(all_results, "output_cleaning_applied"),
    "stalled_loop_rate": rate([r["stalled_loop_detected"] for r in all_results]),
    "legacy_report_tool_used_rate": rate([("tec_" + "build_report") in r["actual_tool_sequence"] for r in all_results]),
}

payload = {
    "experiment_id": EXPERIMENT_ID,
    "architecture": ARCHITECTURE_MODE,
    "model_name": MODEL_NAME,
    "dataset_ref": DATASET_REF,
    "dataset_path": str(DATASET_PATH),
    "model_config": {
        "model_name": MODEL_NAME,
        "load_in_4bit": LOAD_IN_4BIT,
        "load_in_8bit": LOAD_IN_8BIT,
        "torch_dtype": TORCH_DTYPE,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "do_sample": DO_SAMPLE,
    },
    "agent_config": {
        "max_orchestration_steps": MAX_ORCHESTRATION_STEPS,
        "max_role_steps": MAX_ROLE_STEPS,
        "max_tool_calls_per_role": MAX_TOOL_CALLS_PER_ROLE,
        "max_parse_retries": MAX_PARSE_RETRIES,
        "max_tool_retries": MAX_TOOL_RETRIES,
        "state_feedback_mode": STATE_FEEDBACK_MODE,
    },
    "function_handoff_protocol_version": FUNCTION_HANDOFF_PROTOCOL_VERSION,
    "prompt_revision": PROMPT_REVISION,
    "summary": summary,
    "results": all_results,
    "created_at": datetime.now(timezone.utc).isoformat(),
}

with BATCH_OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(to_jsonable(payload), f, ensure_ascii=False, indent=2)

summary_rows = []
for r in all_results:
    summary_rows.append({
        "preset_id": r["selected_preset"],
        "task_type": r["task_config"]["task_type"],
        "agent_success": r["success"],
        "workflow_completed": r["workflow_completed"],
        "terminal_numeric_artifact_present": r["terminal_numeric_artifact_present"],
        "overall_ok": r["overall_ok"],
        "tool_sequence_match": (r.get("metrics") or {}).get("tool_sequence_match"),
        "actual_tool_sequence": r["actual_tool_sequence"],
        "attempted_tec_tool_sequence": r["attempted_tec_tool_sequence"],
        "role_agent_order": r["role_agent_order"],
        "tool_call_count": len(r["actual_tool_sequence"]),
        "tool_error_count": r["tool_error_count"],
        "final_answer_present": bool((r["agent_result"] or {}).get("answer")),
        "invalid_artifact_handle_count": r["invalid_artifact_handle_count"],
        "multiple_function_blocks_in_single_output_count": r["multiple_function_blocks_in_single_output_count"],
        "multiple_protocol_blocks_in_single_output_count": r["multiple_protocol_blocks_in_single_output_count"],
        "repeated_tool_call_count": r["repeated_tool_call_count"],
        "successful_final_tool_without_return_count": r["successful_final_tool_without_return_count"],
        "data_repeat_after_scope_covered": r["data_agent_repeated_retrieval_after_all_assignment_series_present_count"],
        "math_repeat_after_terminal": r["math_repeated_tool_after_terminal_artifact_count"],
        "math_repeat_after_intermediate": r["math_repeated_tool_after_new_intermediate_artifact_count"],
        "math_assignment_completed_but_not_returned": r["math_assignment_completed_but_not_returned_count"],
        "orchestrator_free_text_ignored_count": r["orchestrator_free_text_ignored_count"],
        "avg_state_character_count": diagnostic_avg([r], "state_character_count"),
        "avg_state_history_items_included": diagnostic_avg([r], "state_history_items_included"),
        "stalled_loop_detected": r["stalled_loop_detected"],
        "failure_reason": r["failure_reason"],
        "answer_preview": r["final_answer_preview"],
    })
print("Saved batch:", BATCH_OUTPUT_PATH)
pd.DataFrame(summary_rows)


## 11. Optional architecture comparison


In [ ]:
def load_optional(path):
    if not path.exists():
        print("Missing:", path)
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


REAL_RUNS_MULTI_ROOT = PROJECT_DIR / "outputs" / "metrics" / "real_runs" / "multi_agent"
comparison_sources = {
    "qwen_single": load_optional(PROJECT_DIR / "outputs" / "metrics" / "real_runs" / "single_agent" / "qwen_single_agent_batch_colab.json"),
    "rule_multi": load_optional(REAL_RUNS_MULTI_ROOT / "experiment_1" / "multi_agent_rule_based_five_task_batch.json"),
    "qwen_multi_old": load_optional(REAL_RUNS_MULTI_ROOT / "experiment_1" / "qwen_multi_agent_batch_colab.json"),
    "typed_v3": load_optional(PROJECT_DIR / "outputs" / "metrics" / "experiment_4" / "qwen_multi_agent_typed_v3_batch_colab.json"),
    "function_handoff_v1": load_optional(REAL_RUNS_MULTI_ROOT / "experiment_5_function_handoff" / "qwen_multi_agent_function_handoff_batch_colab.json"),
    "function_handoff_v2": load_optional(REAL_RUNS_MULTI_ROOT / "experiment_6_function_handoff_v2" / "qwen_multi_agent_function_handoff_v2_batch_colab.json"),
    "function_handoff_v4_current": load_optional(BATCH_OUTPUT_PATH),
}

rows = []
for name, payload in comparison_sources.items():
    summary = (payload or {}).get("summary") or {}
    rows.append({
        "source": name,
        "present": payload is not None,
        "model_name": (payload or {}).get("model_name"),
        "architecture": (payload or {}).get("architecture"),
        "prompt_revision": (payload or {}).get("prompt_revision"),
        "success_rate": summary.get("success_rate"),
        "workflow_completed_rate": summary.get("workflow_completed_rate"),
        "terminal_numeric_artifact_present_rate": summary.get("terminal_numeric_artifact_present_rate"),
        "tool_sequence_match_rate": summary.get("tool_sequence_match_rate"),
        "stalled_loop_rate": summary.get("stalled_loop_rate"),
    })
pd.DataFrame(rows)
